<a href="https://colab.research.google.com/github/avialazraki/DI_Bootcamp/blob/main/Week8_Day2_ExercisesXP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 · Day 2 — Exercises XP: Machine Learning Foundations

**Student submission notebook** | Exercises 1–5  
**Focus:** Loan default prediction, feature selection, model choice, evaluation, and ML paradigms.

> **Important dataset distinction:** The supplied “Loan Predication” dataset commonly has a `Loan_Status` column indicating whether a loan was **approved**, not whether a borrower **defaulted**. These are different targets. The discussion below addresses *default prediction* as requested, while the optional dataset inspection demonstrates what can actually be inferred from the supplied file. No default-model results are claimed without default labels.


## Exercise 1 — Problem statement and data collection plan

### Problem statement
Build a **supervised binary classification** model that estimates the probability that an applicant will default on a loan **within 12 months of origination**, using only information available at the time of the lending decision. A default must be explicitly defined by the lender (for example, 90+ days past due or a charged-off account). The output is a calibrated risk probability used to support human review, not an automatic or unappealable decision.

### Data needed and likely sources

| Data category | Example fields | Potential source | Rationale |
|---|---|---|---|
| Loan terms | Requested amount, interest rate, term, installment | Loan origination system | Measures exposure and payment burden |
| Credit history | Credit score, delinquencies, outstanding debt, utilization | Credit bureaus (with appropriate authorization) | Measures historical repayment risk |
| Financial capacity | Verified income, debt-to-income ratio, employment stability | Applications, payroll/income verification, bank records (with consent) | Measures ability to repay |
| Prior repayment behavior | Previous missed payments, past defaults | Lender servicing records, authorized bureau data | Strong potential risk signals |
| Application details | Application date, purpose, existing obligations | Loan applications | Context for risk and monitoring |
| **Target outcome** | Default within 12 months, outcome observation date | Loan servicing / collections | Required for supervised *default* prediction |

**Collection and governance:** Secure appropriate permissions; minimize personal data; encrypt sensitive records; document retention periods; check legal/fair-lending obligations. Exclude names, IDs, and other direct identifiers from training. Review sensitive or proxy attributes for disparate impact rather than automatically assuming exclusion eliminates bias. Ensure loans have had a complete 12-month outcome window or use a suitable censored-time approach. Split records chronologically and by customer where necessary to avoid leakage. Record missingness, provenance, and data quality.

**Success criteria:** Good recall for true defaults while maintaining manageable false-positive rates, well-calibrated probabilities, stable performance over time, and acceptable fairness across relevant groups. The precise threshold should reflect business costs and human oversight.


## Exercise 2 — Feature selection and model choice

The supplied dataset is named *Loan Predication*. Inspect its actual columns below. A common version contains `Gender`, `Married`, `Dependents`, `Education`, `Self_Employed`, `ApplicantIncome`, `CoapplicantIncome`, `LoanAmount`, `Loan_Amount_Term`, `Credit_History`, `Property_Area`, `Loan_ID`, and `Loan_Status`.

### Feature recommendations

| Feature | Keep? | Reason and caveat |
|---|---|---|
| `Credit_History` | Yes, if legally permitted | Existing credit performance may predict risk; check definition and fairness |
| `ApplicantIncome`, `CoapplicantIncome` | Yes | Repayment capacity; combine into total household income where appropriate |
| `LoanAmount` | Yes | Debt exposure; consider amount-to-income ratio |
| `Loan_Amount_Term` | Yes | Repayment schedule and installment burden |
| `Dependents` | Consider | Household financial commitments; validate incremental value |
| `Self_Employed` | Consider | May relate to income volatility; test rather than assume |
| `Education`, `Property_Area` | Caution | May introduce socioeconomic proxies; evaluate necessity and fairness |
| `Gender`, `Married` | Not default predictors in baseline | Sensitive/proxy concerns; retain separately for permitted fairness auditing |
| `Loan_ID` | No | Identifier with no reliable predictive meaning |
| `Loan_Status` | **Not an input feature** | Outcome of the loan-approval decision; if used as a predictor it can leak the target |

**Critical limitation:** If `Loan_Status` means approved/rejected (`Y`/`N`), the dataset supports an **approval classification** exercise, not actual **default prediction**. A valid default model requires observed repayment outcomes and their dates. Approval-only samples may also suffer selection bias because rejected applicants have no observed repayment outcome.

### Preparation strategy
1. Confirm the target definition and column meanings.
2. Separate features from target **before** preprocessing; split train/validation/test (prefer time-based when dates exist).
3. Impute missing numeric values with training-set medians; categorical values with training-set most-frequent or explicit “Unknown”.
4. One-hot encode categories; scale numeric features for logistic regression (tree models do not require scaling).
5. Add sensible derived variables (total income, loan-to-income ratio), avoiding information created after the application.
6. Use cross-validation pipelines to prevent leakage; assess performance and fairness across groups.


In [1]:
# Optional: inspect the course dataset without pretending it contains default outcomes.
# Requires: pandas (and internet access for the ZIP download).
import io
import zipfile
import urllib.request
import pandas as pd

DATA_URL = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%201/Loan%20Predication.zip"
try:
    with urllib.request.urlopen(DATA_URL, timeout=30) as response:
        zip_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        csv_names = [name for name in zf.namelist() if name.lower().endswith(".csv")]
        print("CSV files:", csv_names)
        if not csv_names:
            raise ValueError("No CSV found in downloaded ZIP")
        # Prefer a training file if present.
        chosen = next((name for name in csv_names if "train" in name.lower()), csv_names[0])
        with zf.open(chosen) as f:
            loan_df = pd.read_csv(f)
    print("Dataset:", chosen, "| shape:", loan_df.shape)
    print("Columns:", loan_df.columns.tolist())
    print("\nMissing values:\n", loan_df.isna().sum().to_string())
    print("\nPreview:\n", loan_df.head().to_string(index=False))
    if "Loan_Status" in loan_df.columns:
        print("\nLoan_Status counts:\n", loan_df["Loan_Status"].value_counts(dropna=False))
        print("Interpretation check: verify whether Loan_Status means approval, not default.")
except Exception as exc:
    print("Dataset inspection skipped:", type(exc).__name__, str(exc))
    print("You can download the ZIP manually and inspect its CSV in Colab.")

CSV files: ['train_u6lujuX_CVtuZ9i (1).csv']
Dataset: train_u6lujuX_CVtuZ9i (1).csv | shape: (614, 13)
Columns: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']

Missing values:
 Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0

Preview:
  Loan_ID Gender Married Dependents    Education Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History Property_Area Loan_Status
LP001002   Male      No          0     Graduate            No             5849                0.0         NaN             360.0             1.0         Urban           Y
LP0010

## Exercise 3 — Training, evaluating and optimizing a loan prediction model

**Model choices:** Start with **regularized logistic regression** for an interpretable baseline and calibrated probabilities. Compare with a **random forest** and **gradient-boosted trees** (e.g., HistGradientBoosting or XGBoost if installed) to capture nonlinear effects and interactions. A dummy majority-class model provides a minimum baseline. For a genuine default model, the label must be observed default—not loan approval.

**Evaluation plan:**
1. Define the target and positive class: `1 = default within 12 months`; check prevalence and censoring.
2. Reserve the most recent fully observed loans as an untouched test set; prevent customer leakage.
3. Build imputation/encoding/scaling into a pipeline; tune hyperparameters using cross-validation on training data.
4. Track the **confusion matrix**, **precision** (share of flagged loans that default), **recall** (share of defaults found), **F1-score**, **PR-AUC** (especially for rare defaults), and **ROC-AUC**. Accuracy alone is misleading under imbalance.
5. Assess **calibration** with reliability curves and Brier score, since the output is a risk probability.
6. Select a decision threshold based on costs of missed defaults vs incorrectly flagging creditworthy applicants; assess capacity for manual review.
7. Check subgroup performance, temporal drift, stability, explainability, and robustness; evaluate the final chosen model **once** on the untouched test set.

**Optimization:** Tune logistic regularization and tree depth/learning rate; use class weights or carefully controlled resampling only within training folds. Avoid optimizing solely for accuracy. Explain decisions, maintain audit trails, and allow appropriate human review.

**Important:** A model trained on the course dataset's approval label would answer “Will this application be approved?” rather than “Will this borrower default?”


## Exercise 4 — Choose a suitable learning approach

| Scenario | Primary approach | Why and example model | Important nuance |
|---|---|---|---|
| Predict future stock prices | **Supervised learning (time-series regression/forecasting)** | Predict a future return or price using historical observations; baselines, lagged-feature regression, or time-series models | Use walk-forward validation; nonstationarity and market uncertainty make predictions unreliable |
| Organize library books by similarity | **Unsupervised learning (clustering)** | Embed book descriptions/topics, then group using K-means or hierarchical clustering | If a reliable set of genre labels already exists, supervised classification may be better; clusters need not match human genres |
| Robot finds a path through a maze | **Reinforcement learning** | Train a policy using rewards for reaching a goal and penalties for inefficient movement; Q-learning for a small discrete maze | If the maze map is known and static, **BFS or A\*** search is often simpler and guarantees a shortest path under suitable assumptions |

**Justification:** Supervised learning uses known outcomes, unsupervised learning discovers structure without labels, and reinforcement learning learns actions through interactions and rewards.


## Exercise 5 — Evaluation strategies for three ML categories

### A. Supervised learning — Logistic regression classifier
- **Method:** Train/validation/test split, stratified cross-validation for stable i.i.d. data, or time-based validation for changing populations. Compare to a dummy baseline.
- **Metrics:** Confusion matrix, accuracy, precision, recall, F1, ROC-AUC, PR-AUC, Brier score and calibration plot.
- **Selection:** Prioritize PR-AUC/recall for rare defaults while controlling false positives and ensuring calibrated probabilities. Choose threshold based on practical costs.
- **Challenges:** Class imbalance, target leakage, biased historical approvals, distribution shift, subgroup disparities, and lack of complete repayment outcomes.

### B. Unsupervised learning — K-means clustering of book descriptions
- **Method:** Vectorize or embed book descriptions; standardize suitable numerical features; compare several values of `k`.
- **Metrics:** Silhouette score (higher usually means better separated clusters), elbow curve of within-cluster sum of squares, Davies–Bouldin index (lower generally better), and cluster stability under resampling.
- **Human validation:** Read representative titles/descriptions and ask librarians whether groups are useful. If external genre labels are available, use adjusted Rand index or normalized mutual information as *additional* validation.
- **Challenges:** No unique ground truth, sensitivity to distance metrics, embeddings and `k`, uneven cluster sizes, and clusters that may be mathematically compact but not meaningful.

### C. Reinforcement learning — Q-learning for maze navigation
- **Method:** Evaluate the trained policy on many held-out maze layouts and random seeds, with exploration disabled or minimized at evaluation time.
- **Metrics:** Mean/median cumulative reward, success rate, steps to goal, excess path length relative to a known shortest path, collision/invalid-action rate, and sample efficiency.
- **Learning behavior:** Plot reward and success rate vs training episodes; examine convergence, variance across seeds, and the effect of exploration rate `epsilon`.
- **Challenges:** Reward hacking, exploration vs exploitation, sparse rewards, expensive training, unsafe behavior, and overfitting to familiar mazes. Success on training mazes does not establish generalization.

### Comparison
Supervised models have labels and direct prediction metrics; unsupervised models require structural scores plus human judgment; reinforcement learning needs repeated interactive trials, safety checks and policy-performance measurements.


## Final reflection

These exercises demonstrate that **correct problem framing matters as much as algorithm selection**. In particular, predicting *loan approval* from `Loan_Status` is not equivalent to predicting *loan default*. A credible project must first obtain actual repayment outcomes, protect against leakage, and evaluate not only discrimination but also probability calibration, fairness, and stability.

**What I learned:** how to match ML approaches to available labels and feedback, select meaningful features, choose suitable evaluation metrics, and identify limitations before deployment.

**Practical next step:** obtain an appropriately governed dataset with verified default outcomes and timestamps, build a leakage-safe baseline pipeline, and compare it with nonlinear models on an untouched temporal test set.
